In [1]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR2 = "/content/drive/MyDrive/FactLens_Group9/data2"
ARTIFACTS = DATA_DIR2 + "/artifacts"

import os
os.makedirs(ARTIFACTS, exist_ok=True)

!pip install vaderSentiment textblob textstat -q

print("Drive mounted successfully")
print(f"Data directory: {DATA_DIR2}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.3 MB/s eta 0:00:00
Drive mounted successfully
Data directory: /content/drive/MyDrive/FactLens_Group9/data2


# Left vs Right political bias classifier (TF-IDF + Logistic Regression)

This notebook mirrors the **FakeVsReal** workflow (feature extraction, TF-IDF, enhanced logistic regression, evaluation).

**Setup:** create a virtual environment, then run:
`pip install -r requirements.txt`

**Run:** *Run All* cells. Training uses only **Left** and **Right** rows from `news_bias.csv` (Center articles are excluded for binary classification).

**Outputs:** model and vectorizer are saved under `artifacts/`.

**Note:** If TextBlob complains about missing corpora, run once in a terminal: `python -m textblob.download_corpora`


In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from textblob import TextBlob
import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pickle

warnings.filterwarnings("ignore", category=UserWarning)

CSV_PATH = f"{DATA_DIR2}/news_bias.csv"

print("CSV:", CSV_PATH)

CSV: /content/drive/MyDrive/FactLens_Group9/data2/news_bias.csv


In [3]:
# Load and filter to binary Left vs Right
df = pd.read_csv(CSV_PATH)
df = df[df["label"].isin(["Left", "Right"])].copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 50]

print(f"Rows after filtering: {len(df):,}")
print(df["label"].value_counts())


Rows after filtering: 13,366
label
Left     7803
Right    5563
Name: count, dtype: int64


## Feature extraction (same spirit as FakeVsReal Step3b)
VADER sentiment, TextBlob subjectivity, Flesch readability, punctuation counts.


In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

print("Extracting hand-crafted features...")
analyzer = SentimentIntensityAnalyzer()

def sentiment_compound(t: str) -> float:
    return analyzer.polarity_scores(t)["compound"]

def subjectivity(t: str) -> float:
    return TextBlob(t).sentiment.subjectivity

def readability(t: str) -> float:
    try:
        return float(textstat.flesch_reading_ease(t))
    except Exception:
        return np.nan

texts = df["text"]

print(f"\n[1/5] Extracting sentiment ({len(texts):,} articles)...")
df["sentiment"] = texts.progress_map(sentiment_compound)

print(f"\n[2/5] Extracting subjectivity ({len(texts):,} articles)...")
df["subjectivity"] = texts.progress_map(subjectivity)

print(f"\n[3/5] Extracting readability ({len(texts):,} articles)...")
df["readability"] = texts.progress_map(readability)

print("\n[4/5] Extracting punctuation features...")
df["exclamation_count"] = texts.str.count("!")
df["question_count"] = texts.str.count(r"\?")
print("Done")

print("\n[5/5] Handling missing values...")
df["readability"] = df["readability"].fillna(df["readability"].median())
print("Done")

feature_cols = ["sentiment", "subjectivity", "readability", "exclamation_count", "question_count"]
extra = df[feature_cols].values

print("\nFeatures extracted — scaler will be fit after split to avoid data leakage")
print("Feature matrix shape:", extra.shape)

Extracting hand-crafted features...

[1/5] Extracting sentiment (13,366 articles)...


  0%|          | 0/13366 [00:00<?, ?it/s]

In [ ]:
# Optional: feature distributions by label (like FakeVsReal Step3b)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
plot_feats = ["sentiment", "subjectivity", "readability", "exclamation_count"]
for ax, col in zip(axes.ravel(), plot_feats):
    sns.boxplot(data=df, x="label", y=col, hue="label",
                ax=ax, palette=["#4C72B0", "#C44E52"], legend=False)
    ax.set_title(col)
plt.suptitle("Hand-crafted features by label", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/feature_distributions.png", dpi=120)
plt.show()

In [ ]:
print("Building TF-IDF matrix...")
vec = TfidfVectorizer(
    max_features=40_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

le = LabelEncoder()
y = le.fit_transform(df["label"])
print("Label encoding:", list(zip(le.classes_, range(len(le.classes_)))))
print("TF-IDF will be fit after split to avoid data leakage")

In [ ]:
import re

# Source names and format artifacts identified from coefficient analysis
source_bias_patterns = [
    # Left-leaning sources
    r"nytimes", r"huffpost", r"huffington", r"politico", r"\bvox\b",
    r"msnbc", r"\bcnn\b", r"wapo", r"washingtonpost", r"\bnpr\b",
    r"theatlantic", r"guardian", r"buzzfeed", r"\bslate\b", r"\bsalon\b",
    # Right-leaning sources
    r"breitbart", r"foxnews", r"dailywire", r"dailycaller",
    r"townhall", r"washingtonexaminer", r"washingtontimes",
    r"\bcbn\b", r"\bepoch\b", r"epochtimes", r"\bnr\b", r"nationalreview",
    # Format artifacts
    r"imagecredit", r"creditcredit", r"rightsreserved",
    r"copyright\s*\d{4}", r"newsletter", r"click\s*here", r"now\s*listen",
    r"contributing\s+writer", r"contributed\s+to", r"read\s+more",
    r"top\s+stories", r"pictured\s+above", r"all\s+rights\s+reserved",
    r"news\s+articles",
]

# Combine all patterns into one regex
combined_pattern = re.compile("|".join(source_bias_patterns), re.IGNORECASE)

def remove_source_bias(text):
    return combined_pattern.sub("", str(text)).strip()

print("Removing source bias words using regex...")
df["debiased_text"] = df["text"].apply(remove_source_bias)

# Check average character reduction
before_chars = df["text"].str.len().mean()
after_chars = df["debiased_text"].str.len().mean()
before_words = df["text"].apply(lambda x: len(str(x).split())).mean()
after_words = df["debiased_text"].apply(lambda x: len(str(x).split())).mean()

print(f"Avg chars  before: {before_chars:.0f} → after: {after_chars:.0f} ({before_chars-after_chars:.0f} removed)")
print(f"Avg words  before: {before_words:.0f} → after: {after_words:.0f} ({before_words-after_words:.0f} removed)")

# Verify that key bias terms were actually removed
print("\nVerifying removal:")
for pattern in [r"nytimes", r"huffpost", r"breitbart", r"imagecredit", r"now\s*listen"]:
    before = df["text"].str.contains(pattern, case=False, regex=True).sum()
    after = df["debiased_text"].str.contains(pattern, case=False, regex=True).sum()
    print(f"  '{pattern}': {before:,} articles → {after:,} articles")

In [ ]:
idx = np.arange(len(df))
idx_train, idx_test, y_train, y_test = train_test_split(
    idx, y, test_size=0.2, random_state=42, stratify=y
)

# Fit TF-IDF ONLY on train — using debiased text
print("Fitting TF-IDF on debiased training set only...")
X_tfidf_train = vec.fit_transform(df["debiased_text"].iloc[idx_train])
X_tfidf_test = vec.transform(df["debiased_text"].iloc[idx_test])
print(f"TF-IDF shape: {X_tfidf_train.shape}")

# Scale features ONLY on train
extra_train = extra[idx_train]
extra_test = extra[idx_test]

scaler = StandardScaler()
extra_train_scaled = scaler.fit_transform(extra_train)
extra_test_scaled = scaler.transform(extra_test)

# Combine TF-IDF + features
X_train = sp.hstack([X_tfidf_train, sp.csr_matrix(extra_train_scaled)])
X_test = sp.hstack([X_tfidf_test, sp.csr_matrix(extra_test_scaled)])

print(f"Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}")
print("Train labels:", np.bincount(y_train))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

print("Running grid search with L1 regularization...")
print("L1 forces weak coefficients to zero — reduces overfitting\n")

param_grid = {"C": [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}

grid = GridSearchCV(
    LogisticRegression(
        penalty="l1",          # L1 instead of L2 — more aggressive against overfitting
        solver="liblinear",    # only solver that supports L1 penalty
        max_iter=2000,
        random_state=42,
        class_weight="balanced",
    ),
    param_grid,
    cv=5,
    scoring="accuracy",
    verbose=1
)

grid.fit(X_train, y_train)

print(f"\nBest C value:     {grid.best_params_['C']}")
print(f"Best CV accuracy: {grid.best_score_:.4%}")
print("\nAll results:")
for mean, std, params in zip(
    grid.cv_results_["mean_test_score"],
    grid.cv_results_["std_test_score"],
    grid.cv_results_["params"]
):
    print(f"  C={params['C']:>6} → accuracy: {mean:.4%} ± {std:.4%}")

# Retrain with best C found by grid search
best_C = grid.best_params_["C"]
print(f"\nRetraining with optimal C={best_C} and L1...")
clf = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=best_C,
    max_iter=2000,
    random_state=42,
    class_weight="balanced",
)
clf.fit(X_train, y_train)

# Show how many coefficients L1 zeroed out
n_zero = (clf.coef_[0] == 0).sum()
n_total = len(clf.coef_[0])
print(f"\nCoefficients zeroed out by L1: {n_zero:,} / {n_total:,} ({n_zero/n_total:.1%})")
print("Done.")

## Metrics (accuracy, precision/recall/F1, confusion matrix, ROC-AUC)


In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Accuracy:  {acc:.4f} ({acc:.2%})")
print(f"Macro F1:  {macro_f1:.4f}")
print()
target_names = list(le.classes_)
print(classification_report(y_test, y_pred, target_names=target_names))

if len(le.classes_) == 2:
    pos_idx = 1
    auc = roc_auc_score(y_test, y_prob[:, pos_idx])
    print(f"ROC-AUC (positive={le.classes_[pos_idx]}): {auc:.4f}")

# Overfitting check
train_acc = accuracy_score(y_train, clf.predict(X_train))
gap = train_acc - acc
print(f"\nOverfitting Check:")
print(f"  Training accuracy: {train_acc:.4%}")
print(f"  Test accuracy:     {acc:.4%}")
print(f"  Gap:               {gap:.4%}")
if gap < 0.05:
    print("  ✓ No significant overfitting detected")
elif gap < 0.10:
    print("  ⚠ Minor overfitting present")
else:
    print("  ✗ Significant overfitting detected")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix — Left vs Right")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/confusion_matrix_lr.png", dpi=150)
plt.show()

In [ ]:
if len(le.classes_) == 2:
    pos_idx = 1
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, pos_idx])
    auc = roc_auc_score(y_test, y_prob[:, pos_idx])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC — positive class: {le.classes_[pos_idx]}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{ARTIFACTS}/roc_curve_lr.png", dpi=150)
    plt.show()

In [ ]:
with open(f"{ARTIFACTS}/left_right_logreg.pkl", "wb") as f:
    pickle.dump(clf, f)
with open(f"{ARTIFACTS}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)
with open(f"{ARTIFACTS}/feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open(f"{ARTIFACTS}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Saved:")
print(f"  left_right_logreg.pkl (C={best_C})")
print(f"  tfidf_vectorizer.pkl")
print(f"  feature_scaler.pkl")
print(f"  label_encoder.pkl")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Build feature name list: first 40,000 are TF-IDF words, last 5 are linguistic features
tfidf_feature_names = vec.get_feature_names_out()
extra_names = ["sentiment", "subjectivity", "readability", "exclamation_count", "question_count"]
all_feature_names = list(tfidf_feature_names) + extra_names

coef = clf.coef_[0]

# Create dataframe sorted from highest (Right) to lowest (Left)
coef_df = pd.DataFrame({
    "word": all_feature_names,
    "coefficient": coef
}).sort_values("coefficient", ascending=False)

# L1 zeroes out irrelevant features — only show non-zero ones
nonzero_df = coef_df[coef_df["coefficient"] != 0]
print(f"Non-zero features after L1: {len(nonzero_df):,} / {len(coef_df):,}")

# Top 20 Left indicators (most negative coefficients)
top_left = coef_df[coef_df["coefficient"] < 0].tail(20).sort_values("coefficient")
print("\nTOP 20 LEFT INDICATORS:")
print("=" * 45)
for _, row in top_left.iterrows():
    print(f"  {row['word']:35} {row['coefficient']:+.4f}")

# Top 20 Right indicators (most positive coefficients)
top_right = coef_df[coef_df["coefficient"] > 0].head(20)
print("\nTOP 20 RIGHT INDICATORS:")
print("=" * 45)
for _, row in top_right.iterrows():
    print(f"  {row['word']:35} {row['coefficient']:+.4f}")

# Combined bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

ax1.barh(top_left["word"], top_left["coefficient"], color="#4C72B0")
ax1.set_title("Top 20 Left Indicators", fontsize=13, fontweight="bold")
ax1.set_xlabel("Coefficient Value", fontsize=11)
ax1.axvline(x=0, color="black", linewidth=0.8)

ax2.barh(top_right["word"][::-1], top_right["coefficient"][::-1], color="#C44E52")
ax2.set_title("Top 20 Right Indicators", fontsize=13, fontweight="bold")
ax2.set_xlabel("Coefficient Value", fontsize=11)
ax2.axvline(x=0, color="black", linewidth=0.8)

plt.suptitle("L1 Logistic Regression — Word Importance by Political Bias",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{ARTIFACTS}/left_right_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()